# Analysis routine to compute D0 and compare that with the disdrometer data
## To run this code, you will need the output files below:

**main files**: test_rank0_iter???_mu_t_list_all.npz <br>
**ancillary files**: test_rank0_iter???_model_PhiDP_offset_long, test_rank0_iter???_model_PhiDP_offset_short, test_rank0_iter???_angle_offset.txt, test_rank0_iter???_maskall.npz, test_rank0_iter???_obsdata.npz

## adjust the parameters below to the path where you save your data.

In [ ]:
FOLDER = f"../../MPPAWR_data/250313_test/cond1/" # Directory for the NN output: files such as test_rank0_iter???_mu_t_list_all.npz
path_datetime_Kumagaya = "../txtfiles_datetime/202206_datetime_Kumagaya.txt" # Kumagaya radar datetime when Kumagaya is raining
lookup_tables_path = "../lookup_tables" # Directory for the lookup tables
disdrometer_data_dir = "../../../../20250106_PFN/data" # Directory for disdrometer data

In [ ]:
import os
import torch
import numpy as np
from einops import rearrange
from scipy.interpolate import griddata
from torch.distributions import Gamma
import matplotlib.pyplot as plt
import io
from datetime import datetime

datetime_Kumagaya = np.loadtxt(path_datetime_Kumagaya, dtype=str)

def output_coordinate(angle_offset, el_angle):
    # we will output a 300*800 tensor that contains coordinate information
    X = np.zeros((300, 800))
    Y = np.zeros((300, 800))
    angles = np.array([90-(angle_offset+1.2*i) for i in range(300)])
    for i in range(800):
        X[:, i] = (i+1)*0.075*np.cos(angles*np.pi/180)*np.cos(el_angle*np.pi/180)
        Y[:, i] = (i+1)*0.075*np.sin(angles*np.pi/180)*np.cos(el_angle*np.pi/180)
        
    return X, Y

def load_lookups():
    table_mu_AH = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_AH_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_AV = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_AV_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_kdp = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_kdp_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_ZH = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_ZH_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_Zdr = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_Zdr_el5.txt"), device="cpu", dtype=torch.float32)
    return [table_mu_AH*(0.075*2), table_mu_AV*(0.075*2), table_mu_kdp*(0.075*2), table_mu_ZH, table_mu_Zdr]

def torch_interp1d(x, xp, yp):
    #print(f"{x.device=}, {xp.device=}")                                                                                                               
    idx_left = torch.searchsorted(xp, x, right=True) - 1
    idx_right = idx_left + 1
    slopes = (yp[idx_right] - yp[idx_left]) / (xp[idx_right] - xp[idx_left])
    interpolated = yp[idx_left] + slopes * (x - xp[idx_left])
    return interpolated

def torch_interp2d(x, y, xp, yp, zp):
    #print(f"{x.device=}, {xp.device=}")                                                                                                      
    idx_left = torch.searchsorted(xp, x, right=True) - 1
    idx_right = idx_left + 1
    idx_lower = torch.searchsorted(yp, y, right=True) - 1
    idx_upper = idx_lower + 1
    w_x = (x-xp[idx_left])/(xp[idx_right]-xp[idx_left])
    w_y = (y-yp[idx_lower])/(yp[idx_upper]-yp[idx_lower])
    interpolated = w_x*w_y*zp[idx_right, idx_upper] + (1-w_x)*w_y*zp[idx_left, idx_upper] + w_x*(1-w_y)*zp[idx_right, idx_lower] + (1-w_x)*(1-w_y)*zp[idx_left, idx_lower]
    return interpolated

def Maki(x):
    x_cm = x * 0.1
    # Apply conditions for both cases
    condition = (x_cm < 0.11) | (x_cm > 0.44)
    # Define the two cases
    result_case1 = 1.0048 + 0.0057*x_cm - 2.628*x_cm**2 + 3.682*x_cm**3 - 1.677*x_cm**4
    result_case2 = 1.0048 + 0.0057*x_cm - 2.628*x_cm**2 + 3.682*x_cm**3 - 1.677*x_cm**4
    # Use np.where to choose the correct case
    return torch.where(condition, result_case1, result_case2)

def kDP_function(z, N0_range, loc_range, log_scale_range, lookup_tables):
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    N0s = torch.sigmoid(z[..., 0]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    xp = torch.linspace(-1.2, 1.2, 61, device=z.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z.device)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    return kDP

def observation_DSD_Zh(z, N0_range, loc_range, log_scale_range, lookup_tables):
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    CZH = 0
    # assuming z contains 3 numbers per ot. 3 numbers are ZH, ZV, KDP                                                                                                                            
    N0s = torch.sigmoid(z[..., 0]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    
    xp = torch.linspace(-1.2, 1.2, 61, device=z.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z.device)
    Zh = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_ZH) + 10*N0s
    Zdr = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_Zdr)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    AH = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AH) * pow(10, N0s)
    AV = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AV) * pow(10, N0s)
    
    PhiDP = z[..., 3]                                                                                                                                                           
    PIAH = z[..., 4]
    PIAV = z[..., 5]
    
    ZH = Zh - PIAH
    ZDR = Zdr - (PIAH-PIAV)
                                                                                                                                             
    return torch.stack([ZH, ZDR, PhiDP], dim=-1)

def Dynamics_allsteps(z_half, N0_range, loc_range, log_scale_range, lookup_tables):
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    N0s = torch.sigmoid(z_half[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z_half[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z_half[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    
    xp = torch.linspace(-1.2, 1.2, 61, device=z_half.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z_half.device)
    Zh = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_ZH) + 10*N0s
    Zdr = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_Zdr)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    AH = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AH) * pow(10, N0s)
    AV = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AV) * pow(10, N0s)
                                                                                                 
    PIAH = torch.cumsum(AH, dim=-2)
    PIAV = torch.cumsum(AV, dim=-2)
    PhiDP = torch.cumsum(kDP, dim=-2)
    return torch.concat([z_half[..., 0:1], z_half[..., 1:2], z_half[..., 2:3], PhiDP, PIAH, PIAV], dim=-1)

def N0(z, N0_range=[0, 5]):
    N0s = torch.sigmoid(z[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    return N0s

def loc(z, loc_range):
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    return locs

def scale(z, loc_range, log_scale_range):
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scale_correctionlist = torch.sigmoid(z[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    log_scale = log_scale_correctionlist-0.8729+0.0291*locs-0.0873*locs**2-0.0442*locs**3-0.0925*locs**4
    scalelist = torch.exp(log_scale)
    return scalelist

def D0(z, loc_range, log_scale_range):
    locs = loc(z, loc_range)
    scales = scale(z, loc_range, log_scale_range)
    return torch.exp(locs+3*scales*scales)

def R_function(z, CR, N0_range=[0, 5], loc_range=[4, 10], log_scale_range=[-1, 1]):
    N0s = torch.sigmoid(z[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-loc_range[0])+log_scale_range[0]
    R = pow(10, N0s+CR)*torch.exp(torch.lgamma(4.67+locs))*pow(scales, -(4.67+locs))
    return R

lookup_tables = load_lookups()


In [ ]:
Kumagaya_coord = [-20.469766777364697, 32.122450968084027]

zdr = 4
epoch = 5

timestamps = datetime_Kumagaya

lookup_tables = load_lookups()

Ndata_per_rank = 1363
Dlist = []
N0list = []
loclist = []
scalelist = []
for rank in range(0, 1): # Number of ranks you have used to produce the test data: usually just one
    for i in range(Ndata_per_rank):
        if i in [175, 1103]: # data is corrupted
            Dlist.append(np.zeros(11))
            loclist.append(np.zeros(11))
            scalelist.append(np.zeros(11))
        else:
            idx_time = rank*Ndata_per_rank+i
            mu_t_list_all = np.load(FOLDER+f"test_rank{rank}_iter{i}_mu_t_list_all.npz")["arr_0"]
            mu_t_list_all_full = Dynamics_allsteps(z_half=torch.tensor(mu_t_list_all),
                                                   N0_range=[0, 4],
                                                   loc_range=[-1.0, 1.0],
                                                   log_scale_range=[-0.05, 0.05],
                                                   lookup_tables=lookup_tables)
            mu_t_list_all_output = rearrange(mu_t_list_all_full, "a b c -> (a b) c")
            output_flat = observation_DSD_Zh(mu_t_list_all_output,
                                             N0_range=[0, 4],
                                             loc_range=[-1.0, 1.0],
                                             log_scale_range=[-0.05, 0.05], lookup_tables=lookup_tables)
            output = rearrange(output_flat, "(a b) c -> a b c", a=len(mu_t_list_all)).detach().cpu().numpy()
            timestamp = timestamps[idx_time]
            date = timestamp[2:8]
            obsdata = torch.tensor(np.load(FOLDER+f"test_rank{rank}_iter{i}_obsdata.npz")["arr_0"]).detach().cpu().numpy()
            attenuation = np.load(FOLDER+f"test_rank{rank}_iter{i}_h_attenuation.npz")["arr_0"].reshape(3300, 800)
            mask = torch.tensor(np.load(FOLDER+f"test_rank{rank}_iter{i}_maskall.npz")["arr_0"]).detach().cpu().numpy()
            angle_offset = np.loadtxt(FOLDER+f"test_rank{rank}_iter{i}_angle_offset.txt")

            modification_radome = np.zeros_like(output)
            modification_radome[:, :, 0] = -attenuation
            modification_short = np.zeros_like(output)
            modification_short[:, :118, 2] = 1.0
            modification_long = np.zeros_like(output)
            modification_long[:, 118:, 2] = 1.0
            model_PhiDP_offset_short = torch.load(FOLDER+f"test_rank{rank}_iter{i}_model_PhiDP_offset_short", map_location="cpu").detach()
            model_PhiDP_offset_long = torch.load(FOLDER+f"test_rank{rank}_iter{i}_model_PhiDP_offset_long", map_location="cpu").detach()
            output = output + model_PhiDP_offset_short.numpy()*modification_short+model_PhiDP_offset_long.numpy()*modification_long

            kDP = kDP_function(mu_t_list_all_full,
                               N0_range=[0, 4],
                               loc_range=[-1.0, 1.0],
                               log_scale_range=[-0.05, 0.05],
                               lookup_tables=lookup_tables)
            N0s = N0(mu_t_list_all_full, N0_range=[0, 4])
            locs = loc(mu_t_list_all_full, loc_range=[-1.0, 1.0])
            scales = scale(mu_t_list_all_full, loc_range=[-1.0, 1.0], log_scale_range=[-0.05, 0.05])
            D0s = D0(mu_t_list_all_full, loc_range=[-1.0, 1.0], log_scale_range=[-0.05, 0.05])
            Rs = R_function(mu_t_list_all_full, CR=-1)
            D_el = []
            N0_el = []
            loc_el = []
            scale_el = []
            for el in range(11):
                elevation = el*0.5
                X, Y = output_coordinate(angle_offset, elevation)
                points = np.array([X, Y])
                points_rearranged = rearrange(points, "a b c -> (b c) a")

                values = D0s[el*300:(el+1)*300, :, 0].detach()
                values_rearranged = rearrange(values, "a b -> (a b)")
                cond1 = points_rearranged[:, 0] > Kumagaya_coord[0]-5
                cond2 = points_rearranged[:, 0] < Kumagaya_coord[0]+5
                cond3 = points_rearranged[:, 1] > Kumagaya_coord[1]-5
                cond4 = points_rearranged[:, 1] < Kumagaya_coord[1]+5

                points_around = points_rearranged[cond1*cond2*cond3*cond4]
                values_around = values_rearranged[cond1*cond2*cond3*cond4]
                grid_z0 = griddata(points_around, values_around, Kumagaya_coord, method='cubic')
                D_el.append(grid_z0[0])
                print("D0=", grid_z0[0], f"at {timestamp}, el={elevation}")

                values = locs[el*300:(el+1)*300, :, 0].detach()
                values_rearranged = rearrange(values, "a b -> (a b)")
                cond1 = points_rearranged[:, 0] > Kumagaya_coord[0]-5
                cond2 = points_rearranged[:, 0] < Kumagaya_coord[0]+5
                cond3 = points_rearranged[:, 1] > Kumagaya_coord[1]-5
                cond4 = points_rearranged[:, 1] < Kumagaya_coord[1]+5

                points_around = points_rearranged[cond1*cond2*cond3*cond4]
                values_around = values_rearranged[cond1*cond2*cond3*cond4]
                grid_z0 = griddata(points_around, values_around, Kumagaya_coord, method='cubic')
                loc_el.append(grid_z0[0])

                values = scales[el*300:(el+1)*300, :, 0].detach()
                values_rearranged = rearrange(values, "a b -> (a b)")
                cond1 = points_rearranged[:, 0] > Kumagaya_coord[0]-5
                cond2 = points_rearranged[:, 0] < Kumagaya_coord[0]+5
                cond3 = points_rearranged[:, 1] > Kumagaya_coord[1]-5
                cond4 = points_rearranged[:, 1] < Kumagaya_coord[1]+5

                points_around = points_rearranged[cond1*cond2*cond3*cond4]
                values_around = values_rearranged[cond1*cond2*cond3*cond4]
                grid_z0 = griddata(points_around, values_around, Kumagaya_coord, method='cubic')
                scale_el.append(grid_z0[0])

            Dlist.append(D_el)
            loclist.append(loc_el)
            scalelist.append(scale_el)

# Disdrometer

In [ ]:
NDlist = []
flaglist = []
timelist = []
rainrates = []

vel = np.array([0.050,0.150,0.250,0.350,0.450,0.550,0.650,0.750,0.850,0.950,1.100,
                1.300,1.500,1.700,1.900,2.200,2.600,3.000,3.400,3.800,4.400,5.200,
                6.000,6.800,7.600,8.800,10.400,12.000,13.600,15.200,17.600,20.800])
diameter = np.array([0.062,0.187,0.312,0.437,0.562,0.687,0.812,0.937,1.049,1.170,
                     1.350,1.587,1.822,2.054,2.283,2.623,3.066,3.500,3.923,4.337,
                     4.938, 5.705,6.433,7.123,7.774,8.682,9.769,10.722,11.552,
                     12.270,13.026,13.757])
spread = np.array([0.125,0.125,0.125,0.125,0.125,0.125,0.125,0.125,0.121,0.120,
                   0.239,0.236,0.233,0.231,0.228,0.449,0.438,0.429,0.418,0.409,
                   0.787,0.747,0.709,0.670,0.633,1.157,1.019,0.890,0.772,0.666,
                   0.824,0.647])
for month in ["06"]:
    for day in range(1, 31): 
        print(f"{day=}")
        rain_rate = []     # 降水強度 (mm h^-1)
        num_particles = [] # 全数濃度 (mm^-3)
        _base_time = []    # 時刻 (JST を想定)
        nd = []  # N(D) 粒径分布 (mm^-1 m^-3)
        raw = [] # Parsivel で観測されたスペクトルデータ
        if month == "06" and day == 31:
            continue
        with io.open(f'{disdrometer_data_dir}/d2_2022{month}{day:02d}.csv', encoding="utf-8") as f:
            for line in f:
                line = line.rstrip(";")
                #print(line)
                for i in range(len(line.split(";"))):
                    iline = line.split(";")[i]
                    #print(str(i) + ": " + str(iline))
                    if i == 0:
                        # Date
                        date_tuple = iline.split("/")
                    elif i == 1:
                        # Time
                        time_tuple = iline.split(":")
                        #print(datetime(year=int(date_tuple[0]), month=int(date_tuple[1]), day=int(date_tuple[2]), hour=int(time_tuple[0]), minute=int(time_tuple[1]), second=int(time_tuple[2])))
                        _base_time.append(datetime(year=int(date_tuple[0]), month=int(date_tuple[1]), day=int(date_tuple[2]), hour=int(time_tuple[0]), minute=int(time_tuple[1]), second=int(time_tuple[2])))
                    elif i == 2:
                        # Rainfall Intensity (OTT)
                        rain_rate.append(float(iline)) # unit: mm h^-1
                    elif i == 10:
                        # Number of particle
                        num_particles.append(float(iline))
                    elif i == 14:
                        # Parsivel で観測されたスペクトルデータ
                        if iline == "<SPECTRUM>":
                            l = list(line.split(";")[i:i+1+1023])
                            ll = [s.replace('<SPECTRUM>', '0') for s in l]
                            ss = [s.zfill(1) for s in ll]
                            #print(np.reshape(list(map(int, ss)), (32,32)).T)
                            raw.append(list(map(int, ss)))
                        else:
                            raw.append(list(map(int, [0 for _ in range(1024)])))
                        break

        # 粒形分布の確認
        already = []
        for it in range(len(_base_time)):
            if _base_time[it] in already:
                #print(f"{_base_time[it]} already processed")
                continue
            else:
                already.append(_base_time[it])
            #print(_base_time[it])
            #hour = int(it//60)
            #minute = int(it%60)
            #plt.figure()
            for k in [it]:
                # 粒径分布 計算
                flag = 0
                raw_data = np.reshape(raw[k], (32,32))

                ND = np.array([0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0])
                for j in range(32): # V loop
                    for i in range(32): # D loop
                        ND[j] += raw_data[i][j] / (vel[i] * (0.18 * 0.03) * 60.0) # Cij / (Vi * razer sheet area * sampling time)
                    ND[j] /= spread[j] # Cij / (Vi * razer sheet area * sampling time * delta Di)

                NT = 0
                for j in range(32):
                    NT += ND[j]
                #print(ND)
                #plt.plot(diameter, np.ma.masked_where(ND < 0.1, ND), linestyle=':')
                if ((NT > 10) and (rain_rate[k] > 1)):
                    flag = 1
                    #plt.plot(diameter, np.ma.masked_where(ND < 0.1, ND), linestyle=':')
                    
            flaglist.append(flag)
            NDlist.append(ND)
            timelist.append(_base_time[it])
            rainrates.append(rain_rate[it])

# pick up datetimes that correspond to MPPAWR snapshots

In [ ]:
NDlist_compare = np.array(NDlist)[np.array(flaglist)>0]
D0all_histogram = []
cumsum_all = np.cumsum(NDlist_compare[:, :24]*np.array(diameter[:24]*diameter[:24]*diameter[:24]*spread[:24]), axis=1)
for i in range(1363):
    X = torch.linspace(0.01, 12.0, 1200)
    cumsum = cumsum_all[i]
    total = cumsum[-1]
    cross_pt = list(cumsum < (total/2)).index(False) - 1
    slope = (cumsum[cross_pt + 1] - cumsum[cross_pt]) / (
        diameter[cross_pt + 1] - diameter[cross_pt]
    )
    run = (0.5 * cumsum[-1] - cumsum[cross_pt]) / slope
    D0all_histogram.append(diameter[cross_pt] + run)
D0all_histogram = np.array(D0all_histogram)

In [ ]:
plt.plot(D0all_histogram, label="disdrometer")
plt.plot(np.array(Dlist)[:, 5], label="radar")
plt.xlim(0, 1400)
plt.ylim(0, 5)
plt.legend()
plt.savefig("timeseries_all")

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ax.scatter(D0all_histogram, np.array(Dlist)[:1363, 5], s=1)
ax.set_aspect("equal")
ax.set_xlim(0.3, 5)
ax.set_ylim(0.3, 5)
ax.set_xlabel("disdrometer D0[mm]")
ax.set_ylabel("radar D0[mm]")
fig.savefig("scatter_all")